This experiment is dedicated to test $\hat R_{\nu}$ on differnt bivariate normal distribution, where their covariance differs.

In [1]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 149.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 MB 28.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.


### **Package Import and other Setups**

In [1]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [3]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/JHU Stuff/Capstone/Utility_Functions/BJX_Utility.py'
with open(utility_link) as f: exec(f.read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Hyperparam Setups**

In [4]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [5]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

### **Experiment Part**

### **Isotropic Diagonal Bivariate MVN with rotation**

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  1 & \rho \\
  \rho & 1 \\
\end{bmatrix}
$$

**$\rho=0.3$**:

In [6]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.3
cov = jnp.array([
    [1.0, rho],
    [rho, 1.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [7]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [8]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1513.5 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0003945356293115765
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00031876866705715656
Simulation Start: 1767.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0005520757986232638
Naive initialization. Warmup Length: 20; mean of MSE is: 0.000569471507333219
Simulation Start: 1772.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0007268766057677567
Naive initialization. Warmup Length: 30; mean of MSE is: 0.000616035598795861
Simulation Start: 1773.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005833248142153025
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007237005629576743
Simulation Start: 1774.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00031896348809823394
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0003432332305237651
Simulation Start: 1779.7 MB
Constrained 

**Save to dataframe and file**

In [9]:
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

In [10]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_3_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_3_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_3_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_3_Rhat_n.pkl"
)

**$\rho=0.6$**:

In [11]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.6
cov = jnp.array([
    [1.0, rho],
    [rho, 1.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [12]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [13]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1772.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.007753681391477585
Naive initialization. Warmup Length: 10; mean of MSE is: 0.00015761285612825304
Simulation Start: 1797.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.000547951553016901
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0006078197038732469
Simulation Start: 1781.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0008792093140073121
Naive initialization. Warmup Length: 30; mean of MSE is: 0.000809245917480439
Simulation Start: 1796.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004922159714624286
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005346411489881575
Simulation Start: 1789.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0003807678585872054
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0003663685929495841
Simulation Start: 1787.8 MB
Constrained in

**Save to dataframe and file**

In [14]:
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

In [15]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_6_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_6_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_6_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_6_Rhat_n.pkl"
)

**$\rho=$0.9**

In [16]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.9
cov = jnp.array([
    [1.0, rho],
    [rho, 1.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [17]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [18]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1797.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.02111603133380413
Naive initialization. Warmup Length: 10; mean of MSE is: 0.13956113159656525
Simulation Start: 1813.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0008737159660086036
Naive initialization. Warmup Length: 20; mean of MSE is: 0.011209717020392418
Simulation Start: 1793.8 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0007391631952486932
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0010327397612854838
Simulation Start: 1786.5 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005301979836076498
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007128026918508112
Simulation Start: 1800.0 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.000452160689746961
Naive initialization. Warmup Length: 50; mean of MSE is: 0.000724179029930383
Simulation Start: 1802.2 MB
Constrained initial

**Save to dataframe and file**

In [19]:
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

In [20]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_9_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_9_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_9_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1_9_Rhat_n.pkl"
)

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  10 & \rho*10 \\
  \rho*10 & 10 \\
\end{bmatrix}
$$

**$\rho=0.3$**:

In [21]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.3
cov = jnp.array([
    [10.0, 10*rho],
    [10*rho, 10.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [22]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [23]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1814.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.000652475340757519
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005501260748133063
Simulation Start: 1813.3 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0003269122389610857
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00041248518391512334
Simulation Start: 1814.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004659316036850214
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0004803320043720305
Simulation Start: 1813.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.000521671143360436
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00042351591400802135
Simulation Start: 1811.5 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00048010703176259995
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005078028771094978
Simulation Start: 1810.5 MB
Constrained

In [24]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_3_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_3_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_3_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_3_Rhat_n.pkl"
)

**$\rho = 0.6$**

In [25]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.6
cov = jnp.array([
    [10.0, 10*rho],
    [10*rho, 10.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [26]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [27]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1799.8 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.004895365796983242
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0022922074422240257
Simulation Start: 1822.0 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004081553779542446
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0002555932733230293
Simulation Start: 1812.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005528643960133195
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00046098517486825585
Simulation Start: 1807.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.00057172030210495
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005567671032622457
Simulation Start: 1807.7 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0002935614320449531
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00033178305602632463
Simulation Start: 1798.4 MB
Constrained i

In [28]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_6_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_6_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_6_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_6_Rhat_n.pkl"
)

**$\rho=0.9$**

In [29]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.9
cov = jnp.array([
    [10.0, 10*rho],
    [10*rho, 10.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [30]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [31]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1800.9 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.06285028904676437
Naive initialization. Warmup Length: 10; mean of MSE is: 0.031972240656614304
Simulation Start: 1811.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.004995841532945633
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0024810165632516146
Simulation Start: 1815.1 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.000682560377754271
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00043247820576652884
Simulation Start: 1806.3 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0007847541128285229
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0003978954628109932
Simulation Start: 1821.5 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.000455579865956679
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00045667338417842984
Simulation Start: 1806.7 MB
Constrained init

In [32]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_9_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_9_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_9_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/10_9_Rhat_n.pkl"
)

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  100 & \rho*100 \\
  \rho*100 & 100 \\
\end{bmatrix}
$$

**$\rho = 0.3$**

In [33]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.3
cov = jnp.array([
    [100.0, 100*rho],
    [100*rho, 100.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [34]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [35]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1799.1 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.000765179400332272
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005182723980396986
Simulation Start: 1816.2 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0003142514906357974
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0004009637050330639
Simulation Start: 1815.7 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004927768604829907
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005633719847537577
Simulation Start: 1808.8 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005138727137818933
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0004605984431691468
Simulation Start: 1816.8 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0004142001271247864
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004398055316414684
Simulation Start: 1830.2 MB
Constrained i

In [36]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_3_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_3_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_3_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_3_Rhat_n.pkl"
)

**$\rho = 0.6$**

In [37]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.6
cov = jnp.array([
    [100.0, 100*rho],
    [100*rho, 100.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [38]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [39]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1799.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.003966990392655134
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0025656104553490877
Simulation Start: 1816.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0003789328911807388
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00030429361504502594
Simulation Start: 1798.3 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.000499928486533463
Naive initialization. Warmup Length: 30; mean of MSE is: 0.000497793429531157
Simulation Start: 1809.0 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005839627119712532
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0006082718027755618
Simulation Start: 1814.1 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00032135361107066274
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00030839117243885994
Simulation Start: 1807.9 MB
Constrained 

In [40]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_6_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_6_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_6_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_6_Rhat_n.pkl"
)

**$\rho = 0.9$**

In [41]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.9
cov = jnp.array([
    [100.0, 100*rho],
    [100*rho, 100.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [42]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [43]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1816.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.042306818068027496
Naive initialization. Warmup Length: 10; mean of MSE is: 0.02494368515908718
Simulation Start: 1817.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.02515042945742607
Naive initialization. Warmup Length: 20; mean of MSE is: 0.01266102958470583
Simulation Start: 1808.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.010443122126162052
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0050948988646268845
Simulation Start: 1807.1 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.006281075067818165
Naive initialization. Warmup Length: 40; mean of MSE is: 0.002407032297924161
Simulation Start: 1800.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0020308741368353367
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0007909525884315372
Simulation Start: 1823.4 MB
Constrained initializa

In [44]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_9_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_9_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_9_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/100_9_Rhat_n.pkl"
)

#### **Distribution with covariance matrix**:
$$
\begin{bmatrix}
  1000 & \rho*1000 \\
  \rho*1000 & 1000 \\
\end{bmatrix}
$$

**$\rho = 0.3$**

In [45]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.3
cov = jnp.array([
    [1000.0, 1000*rho],
    [1000*rho, 1000.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [46]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [47]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1814.6 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0007773353136144578
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0006453933892771602
Simulation Start: 1816.1 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004154483904130757
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00042725828825496137
Simulation Start: 1802.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.00044768041698262095
Naive initialization. Warmup Length: 30; mean of MSE is: 0.00039448108873330057
Simulation Start: 1815.7 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004478625487536192
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005577464471571147
Simulation Start: 1810.4 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00022057056776247919
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00036413344787433743
Simulation Start: 1813.1 MB
Constra

In [48]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_3_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_3_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_3_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_3_Rhat_n.pkl"
)

**$\rho = 0.6$**

In [49]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.6
cov = jnp.array([
    [1000.0, 1000*rho],
    [1000*rho, 1000.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [50]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [51]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1801.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0018463187152519822
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0012245462276041508
Simulation Start: 1825.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004032286233268678
Naive initialization. Warmup Length: 20; mean of MSE is: 0.00030920992139726877
Simulation Start: 1802.9 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0005155425751581788
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0005056500667706132
Simulation Start: 1810.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0005681389011442661
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0007347004720941186
Simulation Start: 1802.9 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00030071736546233296
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0003980524779763073
Simulation Start: 1825.0 MB
Constraine

In [52]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_6_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_6_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_6_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_6_Rhat_n.pkl"
)

**$\rho = 0.9$**

In [53]:
num_dim = 2
# mean array:
mu = jnp.full(num_dim, 0.0)
# covariance matrix:
rho = 0.9
cov = jnp.array([
    [1000.0, 1000*rho],
    [1000*rho, 1000.0]
])

target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
  offset = 2
  return 10*random.normal(key, shape + (num_dim,)) + offset

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [54]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [55]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1817.3 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0052198064513504505
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0030112331733107567
Simulation Start: 1823.9 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0025763416197150946
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0016505293315276504
Simulation Start: 1809.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.001748239970766008
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0007421752670779824
Simulation Start: 1802.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0010155828204005957
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0005823185201734304
Simulation Start: 1802.6 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0006400668062269688
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0005067799356766045
Simulation Start: 1808.6 MB
Constrained i

In [56]:
# Save to dataframe and file
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

MSE_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_9_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_9_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_9_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/JHU Stuff/Capstone/pkl_Additional_Experiment/GeometryExperiment/Rotation/Bi_Diag/1000_9_Rhat_n.pkl"
)